<a href="https://colab.research.google.com/github/fdx-hw/cosc-650/blob/main/week5_ragas_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAGAS Evaluation of the Week 5 Orbit RAG Pipeline

**Pipeline under evaluation:** the Week 5 pipeline, fixed to the `small (120/20)` chunking
config,
retrieving top-3 chunks and generating with Claude (`claude-haiku-4-5-20251001`). This config
is chosen deliberately rather than the best-scoring one from Week 5, it's the config with
real, already characterized weaknesses, which makes for a more honest evaluation than picking
the config closest to ceiling.

**Methodology note:** the four RAGAS dimensions (faithfulness, answer relevancy, context
precision, context recall) are implemented here directly with Claude as an LLM judge,
following RAGAS's own published definitions (claim decomposition + NLI-style verification for
faithfulness and context recall, generated-question embedding similarity for answer
relevancy, rank-weighted LLM relevance verdicts for context precision) -- rather than
installing the `ragas` package itself, which defaults to OpenAI and would need an extra
LangChain/Anthropic wrapper layer that adds real integration risk for no real benefit here,
since the definitions themselves are what matters.


In [1]:
%pip install -q sentence-transformers faiss-cpu anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 44.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 39.5 MB/s eta 0:00:00


In [2]:
import os, re, json, pathlib, numpy as np
os.environ['HF_HOME'] = str((pathlib.Path('.') / '.hf_cache').resolve())
from sentence_transformers import SentenceTransformer
import faiss
embedder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

DOCS = {
    'networking': '''## Networking
The Orbit server listens on TCP port 7700 by default. Gossip runs on 7711.
Clients should connect to port 7700; port 7711 is for inter-node gossip only
and should not be exposed publicly.''',

    'storage': '''## Storage
Orbit persists data with an LSM-tree storage engine.
Writes go to an in-memory memtable first and are flushed to disk as
immutable SSTable segments once the memtable fills.''',

    'replication': '''## Replication
Each key is replicated with a default replication factor of 3.
Default consistency is quorum, meaning a write or read must be
acknowledged by at least two of the three replicas to succeed.''',

    'limits': '''## Limits
The maximum value size is 16 MB. A batch may contain at most 1000 operations.
Keys are limited to 1 KB. Requests exceeding any of these limits are
rejected with a client-side validation error before reaching storage.''',

    'security': '''## Security
Clients authenticate with API tokens. Traffic is encrypted with TLS.
Tokens are scoped per-namespace and can be revoked instantly through
the admin API without restarting the server.''',

    'backup': '''## Backup
Orbit takes incremental snapshots every 6 hours by default, written to
the configured backup directory. A full snapshot can be triggered manually
with orbitctl snapshot --full. Snapshots older than 7 days are pruned
automatically unless retention is overridden in orbit.yaml.''',

    'monitoring': '''## Monitoring
Orbit exposes Prometheus-compatible metrics on port 7720 at /metrics.
Health checks are available at /healthz on the same port. The metrics
port is separate from the client port and is not encrypted by default.''',

    'deployment': '''## Deployment
Orbit is configured through a single YAML file, orbit.yaml, loaded at
startup. Environment variables prefixed with ORBIT_ override values from
the file. The default data directory is /var/lib/orbit, and the default
config search path is /etc/orbit/orbit.yaml.''',
}
print('doc count:', len(DOCS))

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

doc count: 8


In [3]:
def chunk_fixed(text, size, overlap):
    text = re.sub(r'\s+', ' ', text).strip(); out, i = [], 0
    while i < len(text):
        out.append(text[i:i+size]); i += size - overlap
    return out

chunks, sources = [], []
for name, text in DOCS.items():
    for c in chunk_fixed(text, 120, 20):
        chunks.append(c); sources.append(name)
print(f'small (120/20) -> {len(chunks)} chunks')

_vecs = embedder.encode(chunks, normalize_embeddings=True).astype('float32')
index = faiss.IndexFlatIP(_vecs.shape[1]); index.add(_vecs)

def retrieve(q, k=3):
    qe = embedder.encode([q], normalize_embeddings=True).astype('float32')
    ids = index.search(qe, k)[1][0]
    return [chunks[i] for i in ids if i >= 0]

small (120/20) -> 22 chunks


In [4]:
def _get_api_key():
    try:
        from google.colab import userdata
        key = userdata.get('ANTHROPIC_API_KEY')
        if key:
            return key
    except Exception:
        pass
    key = os.environ.get('ANTHROPIC_API_KEY', '').strip()
    if key:
        return key
    try:
        import getpass
        return getpass.getpass('ANTHROPIC_API_KEY (leave blank to skip): ').strip()
    except Exception:
        return ''

_API_KEY = _get_api_key()
if _API_KEY:
    import anthropic
    _client = anthropic.Anthropic(api_key=_API_KEY)
GEN_MODEL = 'claude-haiku-4-5-20251001'

def claude_chat(prompt, max_tokens=800):
    if not _API_KEY:
        return None
    return _client.messages.create(
        model=GEN_MODEL, max_tokens=max_tokens,
        messages=[{'role': 'user', 'content': prompt}]
    ).content[0].text

def answer(q, ctx):
    prompt = ('Answer only from the context.\nContext:\n- ' + '\n- '.join(ctx) +
              f'\nQuestion: {q}\nAnswer:')
    out = claude_chat(prompt)
    return out if out is not None else '[API-BLOCKED] set ANTHROPIC_API_KEY to generate'

In [5]:
def _judge_json(prompt):
    """Call Claude as a judge and parse a single JSON array/object out of the reply,
    tolerating prose or a code fence around it (models don't always return bare JSON)."""
    resp = claude_chat(prompt)
    if resp is None:
        return None
    start_candidates = [i for i in (resp.find('{'), resp.find('[')) if i != -1]
    if not start_candidates:
        raise ValueError(f'No JSON found in judge response: {resp!r}')
    start = min(start_candidates)
    end = max(resp.rfind('}'), resp.rfind(']')) + 1
    return json.loads(resp[start:end])

def faithfulness(answer_text, context):
    """RAGAS definition: decompose the answer into atomic claims, then check each claim
    against the retrieved context. Score = fraction of claims the context supports."""
    claims = _judge_json(
        'Break the following answer into a JSON list of short, atomic factual claims '
        '(each claim a single self-contained statement, no explanation). Reply with ONLY '
        f'a JSON array of strings.\n\nAnswer: {answer_text}'
    )
    if not claims:
        return 1.0, []
    verdicts = _judge_json(
        'For each claim below, decide if it is directly supported by the context. Reply '
        'with ONLY a JSON array of true/false booleans, same order and length as the '
        f'claims.\n\nContext:\n{context}\n\nClaims:\n' +
        '\n'.join(f'- {c}' for c in claims)
    )
    supported = sum(1 for v in verdicts if v)
    return supported / len(claims), list(zip(claims, verdicts))

def answer_relevancy(question, answer_text, n=3):
    """RAGAS definition: generate n questions the answer would plausibly be responding
    to, then score relevancy as the average embedding similarity to the real question."""
    generated = _judge_json(
        f'Given ONLY this answer (not the original question), write {n} different '
        'questions this answer would plausibly be a response to. Reply with ONLY a JSON '
        f'array of {n} question strings.\n\nAnswer: {answer_text}'
    )
    if not generated:
        return 0.0
    qv = embedder.encode([question], normalize_embeddings=True)[0]
    gv = embedder.encode(generated, normalize_embeddings=True)
    return float((gv @ qv).mean())

def context_precision(question, ctx_list):
    """RAGAS definition: judge each retrieved chunk's relevance to the question in rank
    order, then compute the rank-weighted average precision over the true verdicts."""
    verdicts = _judge_json(
        'For each context chunk below, decide if it is relevant to answering the '
        'question. Reply with ONLY a JSON array of true/false booleans, same order and '
        f'length as the chunks.\n\nQuestion: {question}\n\nChunks:\n' +
        '\n'.join(f'{i+1}. {c}' for i, c in enumerate(ctx_list))
    )
    if not any(verdicts):
        return 0.0
    num, hits = 0.0, 0
    for k, v in enumerate(verdicts, start=1):
        if v:
            hits += 1
            num += hits / k
    return num / hits

def context_recall(ground_truth, ctx_list):
    """RAGAS definition: decompose the reference (ground-truth) answer into claims, then
    check whether each is attributable to the retrieved context. Score = fraction
    attributable -- this is what tells you whether retrieval found everything needed."""
    claims = _judge_json(
        'Break the following reference answer into a JSON list of short, atomic factual '
        f'claims. Reply with ONLY a JSON array of strings.\n\nReference answer: {ground_truth}'
    )
    if not claims:
        return 1.0
    context = '\n'.join(ctx_list)
    verdicts = _judge_json(
        'For each claim below, decide if it can be attributed to (derived from) the '
        'context. Reply with ONLY a JSON array of true/false booleans, same order and '
        f'length as the claims.\n\nContext:\n{context}\n\nClaims:\n' +
        '\n'.join(f'- {c}' for c in claims)
    )
    attributable = sum(1 for v in verdicts if v)
    return attributable / len(claims)

## Queries

Six realistic queries (>= 5 required), each with a reference (ground-truth) answer decided
in advance from the doc text, not adjusted after seeing results. Includes the query from
Week 5 Part 4 with the known chunk-split issue, plus two compound queries (asking for two
facts at once) and three straightforward single-fact queries, for a reasonable spread of
difficulty rather than all easy or all hard.

In [6]:
QUERIES = [
    ('where can i check server health',
     'Health checks are available at /healthz on port 7720.'),
    ('what port does orbit listen on for clients',
     'Orbit listens on TCP port 7700 for client connections.'),
    ('how do clients authenticate',
     'Clients authenticate with API tokens.'),
    ('what is the default replication factor and consistency level',
     'The default replication factor is 3, and the default consistency level is quorum.'),
    ('how often are snapshots taken and how long are they kept',
     'Snapshots are taken every 6 hours by default and are pruned after 7 days unless '
     'retention is overridden.'),
    ('what happens if a value exceeds the size limit',
     'Requests exceeding the size limits are rejected with a client-side validation error '
     'before reaching storage.'),
]
print(f'{len(QUERIES)} queries defined (>= 5 required)')

6 queries defined (>= 5 required)


In [7]:
results = []
for q, ground_truth in QUERIES:
    ctx = retrieve(q, k=3)
    resp = answer(q, ctx)
    f_score, f_detail = faithfulness(resp, '\n'.join(ctx))
    ar_score = answer_relevancy(q, resp)
    cp_score = context_precision(q, ctx)
    cr_score = context_recall(ground_truth, ctx)
    results.append({
        'query': q, 'answer': resp,
        'faithfulness': f_score, 'answer_relevancy': ar_score,
        'context_precision': cp_score, 'context_recall': cr_score,
    })
    print(f'{q!r}')
    print(f'  answer: {resp}')
    print(f'  faithfulness={f_score:.2f}  answer_relevancy={ar_score:.2f}  '
          f'context_precision={cp_score:.2f}  context_recall={cr_score:.2f}')
    print()

print('=== averages ===')
for dim in ('faithfulness', 'answer_relevancy', 'context_precision', 'context_recall'):
    avg = sum(r[dim] for r in results) / len(results)
    print(f'{dim:18s} {avg:.2f}')

'where can i check server health'
  answer: Based on the context, you can check server health at:

1. `/healthz` endpoint on port 7720
2. `/heal` endpoint on port 7720

Both health check endpoints are available on the same metrics port (7720).
  faithfulness=1.00  answer_relevancy=0.82  context_precision=1.00  context_recall=1.00

'what port does orbit listen on for clients'
  answer: According to the context, clients should connect to port 7700, which is the default TCP port that the Orbit server listens on.
  faithfulness=1.00  answer_relevancy=0.82  context_precision=1.00  context_recall=0.50

'how do clients authenticate'
  answer: According to the context, clients authenticate with API tokens.
  faithfulness=1.00  answer_relevancy=0.84  context_precision=1.00  context_recall=1.00

'what is the default replication factor and consistency level'
  answer: # Answer

Based on the context provided:

- **Default replication factor:** 3
- **Default consistency level:** Quorum (requiring a